# **Visualizing Life Expectancy & Fertility (1964–2013)**

**Main Research Question:**
How did the relationship between fertility and life expectancy evolve across countries and continents between 1964 and 2013, and to what extent did countries converge toward lower fertility and higher life expectancy?

This question focuses on both temporal change and structural change. It examines not only how fertility and life expectancy changed independently, but also how the relationship between the two evolved over time.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_csv("gapminder.csv")
df.head()

# inspect data
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())

# delete missing values
df = df.dropna(subset = ["lifeExp", "Fertility", "pop", "ID"])

(12200, 7)
Index(['Country', 'Year', 'lifeExp', 'pop', 'Fertility', 'Region', 'ID'], dtype='str')
Country          str
Year           int64
lifeExp      float64
pop          float64
Fertility    float64
Region           str
ID               str
dtype: object
Country         0
Year            0
lifeExp      2089
pop           394
Fertility    2100
Region          0
ID             50
dtype: int64
               Year       lifeExp           pop     Fertility
count  12200.000000  10111.000000  1.180600e+04  10100.000000
mean    1988.500000     64.078600  2.196105e+07      4.028719
std       14.431461     11.122779  9.593930e+07      2.013968
min     1964.000000      6.000000  5.000000e+01      0.836000
25%     1976.000000     56.282500  2.516718e+05      2.175750
50%     1988.500000     67.157000  3.497884e+06      3.632500
75%     2001.000000     72.484000  1.139999e+07      5.905250
max     2013.000000     83.580000  1.359368e+09      9.223000


## Analyze the Time Trend of Fertility and Life Expectancy

Calculate the average life expectancy and fertility of each year to see the change.

In [3]:
# analyze by year
global_trend = (
    df.groupby("Year").agg(
        ave_lifeExp = ("lifeExp", "mean"),
        ave_Fertility = ("Fertility", "mean")
    )
    .reset_index()
)
global_trend.head()

# calculate the overall change of life expectancy and fertility from 1964 to 2013
start = global_trend.iloc[0]
end = global_trend.iloc[-1]

print("Life expectancy change by:",
      round(end["ave_lifeExp"] - start["ave_lifeExp"],2))
print("Fertility change by:",
      round(end["ave_Fertility"] - start["ave_Fertility"],2))

Life expectancy change by: 14.72
Fertility change by: -2.65


Analyze the correlation coefficient of the two variables

In [4]:
# Analyze the relationship between the two variables using the whole sample
overall_correlation = df["Fertility"].corr(df["lifeExp"])
print("Overall correlation:",round(overall_correlation,2))

# the correlation coefficient is -0.83, really strong negative relation

Overall correlation: -0.83


In [5]:
# calculate the correlation coefficient of each year, regardless of continents and countries
corr_by_year = (
    df.groupby("Year")
    .apply(lambda x: x["Fertility"].corr(x["lifeExp"]))
    .reset_index(name = "Correlation")
)

print(corr_by_year.head(50))

fig = px.line(
    corr_by_year,
    x = "Year",
    y = "Correlation",
    title = "Correlation Between Fertility and Life Expectancy Over Time",
    markers = True
)

fig.add_hline(
    y = -0.70,
    line_dash = "dash",
    line_color = "black"
)

fig.show()

# we can see an obvious decline from 1964 to 1993 and a subtle increase from 1995 to 2013, but in this period the correlation is still strongly negative. We can also observe some outliers, 1975-1979 and 1994

    Year  Correlation
0   1964    -0.733581
1   1965    -0.737633
2   1966    -0.744933
3   1967    -0.754333
4   1968    -0.758072
5   1969    -0.763497
6   1970    -0.775376
7   1971    -0.781954
8   1972    -0.787504
9   1973    -0.789831
10  1974    -0.789870
11  1975    -0.777397
12  1976    -0.774055
13  1977    -0.775092
14  1978    -0.778061
15  1979    -0.784886
16  1980    -0.796251
17  1981    -0.811528
18  1982    -0.821979
19  1983    -0.827981
20  1984    -0.833540
21  1985    -0.837404
22  1986    -0.845601
23  1987    -0.850139
24  1988    -0.853624
25  1989    -0.854909
26  1990    -0.854884
27  1991    -0.854609
28  1992    -0.854107
29  1993    -0.849742
30  1994    -0.813271
31  1995    -0.844186
32  1996    -0.845832
33  1997    -0.843573
34  1998    -0.839094
35  1999    -0.833541
36  2000    -0.829154
37  2001    -0.825393
38  2002    -0.822217
39  2003    -0.819511
40  2004    -0.817615
41  2005    -0.814286
42  2006    -0.813493
43  2007    -0.813311
44  2008  

Calculate country-wide differences (convergence)

In [6]:
dispersion = (
    df.groupby("Year")
    .agg(
        lifeExp_std = ("lifeExp", "std"),
        Fertility_std = ("Fertility", "std")
    )
    .reset_index()
)

dispersion.head(50)

fig1 = px.line(
    dispersion,
    x = "Year",
    y = "lifeExp_std",
    title = "Change of std of life expectancy",
    markers = True
)
fig1.show()

fig2 = px.line(
    dispersion,
    x = "Year",
    y = "Fertility_std",
    title = "Change of std of Fertility",
    markers = True
)
fig2.show()

Recognize the countries with the largest change

In [7]:
country_change = (
    df[df["Year"].isin([1964,2013])]
    .pivot(
        index = "Country",
        columns = "Year",
        values = ["lifeExp","Fertility"])
)

country_change["lifeExp_change"] = (
    country_change[("lifeExp", 2013)] -
    country_change[("lifeExp", 1964)]
)

country_change["fertility_change"] = (
    country_change[("Fertility", 2013)] -
    country_change[("Fertility", 1964)]
)

country_change.sort_values(
    "lifeExp_change",
    ascending=False
).head(10)
#
# country_change.sort_values(
#     "fertility_change",
#     ascending=False
# ).head(10)

lifeExp         Fertility        lifeExp_change  \
Year              1964    2013      1964   2013                  
Country                                                          
Maldives        38.911  77.919     7.179  2.256         39.008   
Bhutan          33.827  68.294     6.670  2.232         34.467   
Timor-Leste     35.724  67.538     6.347  5.855         31.814   
Tunisia         44.903  75.873     7.107  2.008         30.970   
Oman            45.895  76.552     7.263  2.853         30.657   
Cambodia        41.900  71.916     6.909  2.861         30.016   
Nepal           40.071  68.410     5.997  2.300         28.339   
Western Sahara  39.880  67.764     6.562  2.363         27.884   
Saudi Arabia    47.781  75.479     7.257  2.644         27.698   
Afghanistan     33.639  60.947     7.671  4.900         27.308   

               fertility_change  
Year                             
Country                          
Maldives                 -4.923  
Bhutan                   -4.438  
Timor-Leste              -0.492  
Tunisia                  -5.099  
Oman                     -4.410  
Cambodia                 -4.048  
Nepal                    -3.697  
Western Sahara           -4.199  
Saudi Arabia             -4.613  
Afghanistan              -2.771

## Interactive Bubble Chart

Show the movement of countries, categorized by region, on the coordinate system of Fertility and Life Expectancy

In [8]:
from bokeh.plotting import figure, output_file, show
from bokeh.models import ColumnDataSource, Slider, HoverTool, CustomJS
from bokeh.layouts import column

In [9]:
print(sorted(df["Region"].unique().tolist()))

initial_year = 1964

df["bubble_size"] = np.cbrt(df["pop"]) / 10 # to prevent extremely large bubble size due to large population

region_color = {
    "America": "blue",
    "East Asia & Pacific": "red",
    "Europe & Central Asia": "green",
    "Middle East & North Africa": "orange",
    "South Asia": "purple",
    "Sub-Saharan Africa": "yellow"
}

df["color"] = df["Region"].map(region_color)

df_year = df[df["Year"] == initial_year].copy()
source = ColumnDataSource(df_year)

image = figure(
    width = 800,
    height = 600,
    title = f"Life Expectancy vs Fertility({initial_year})",
    x_axis_label = "Fertility (children per woman)",
    y_axis_label = "Life Expectancy (years)",
    tools = "pan, wheel_zoom, box_zoom, reset, save"
)

# draw bubbles
image.scatter(
    x = "Fertility",
    y = "lifeExp",
    size = "bubble_size",
    color = "color",
    source = source,
    fill_alpha = 0.6,
    line_color = "black",
    line_alpha = 0.3,
    legend_field = "Region"
)


# add the hovering tool
hover = HoverTool(tooltips = [
    ("Country", "@Country"),
    ("Year", "@Year"),
    ("Life Expectancy", "@lifeExp{0.0}"),
    ("Population", "@pop{0.0}"),
    ("Fertility", "@Fertility{0.0}"),
    ("Region", "@Region")
])
image.add_tools(hover)

image.legend.location = "bottom_left"

image.legend.click_policy = "hide"

slider = Slider(
    start = 1964,
    end = 2013,
    value = 1964,
    step = 1,
    title = "Year",
)

data_by_year = {
    str(year): df[df["Year"] == year].to_dict(orient = "list")
    for year in sorted(df["Year"].unique())
}

# define the callback code
callback = CustomJS(args=dict(
    source=source,
    slider = slider,
    data = data_by_year,
    plot = image),
    code = """
    const year = slider.value.toString();
    data = data[year];
    source.data = data;
    plot.title.text = "Life Expectancy vs. Fertility (" + year + ")";
"""
)
slider.js_on_change("value", callback)

image.title.text_font_size = "18pt"
image.title.align = "center"

layout = column(slider, image)
output_file("Main visualization.html")
show(layout)

['America', 'East Asia & Pacific', 'Europe & Central Asia', 'Middle East & North Africa', 'South Asia', 'Sub-Saharan Africa']


## Sub-Question 1: Which continent experienced the fastest overall demographic transition between 1964 and 2013?

The fastest demographic transition is defined as the continent that experienced the largest decline in fertility, the largest increase in life expectancy, and the greatest overall movement in the fertility-life expectancy space between 1964 and 2013.
I use transition distance at here, combining the moving distance of the two variables together, D=((ΔF)*2+(ΔL)*2)*0.5. The larger the D, the larger the movement of the country in fertility-life expectany coordinate system.

In [10]:
# analyze by region and year
region_trend = (
    df.groupby(["Year","Region"]).agg(
        lifeExp = ("lifeExp", "mean"),
        Fertility = ("Fertility", "mean")
    )
    .reset_index()
)

pivot = region_trend.pivot(
    index = "Region",
    columns = "Year",
    values = ["Fertility","lifeExp"]
)

pivot.columns = [
    f"{variable}_{year}"
    for variable,year in pivot.columns
]
pivot = pivot.reset_index()

pivot["fertility_change"] = (
    pivot["Fertility_2013"] - pivot["Fertility_1964"]
)
pivot["lifeExp_change"] = (
    pivot["lifeExp_2013"] - pivot["lifeExp_1964"]
)

pivot["transition_distance"] = np.sqrt(
    pivot["fertility_change"]**2
    + pivot["lifeExp_change"]**2
)

pivot = pivot.sort_values(
    "transition_distance",
    ascending = True
)

summary = pivot[
    ["Region",
    "fertility_change",
    "lifeExp_change",
    "transition_distance"]
].round(2)

summary = summary.sort_values(
    by = "transition_distance",
    ascending = False
)

print(summary.to_string(index = True))

                       Region  fertility_change  lifeExp_change  transition_distance
4                  South Asia             -3.75           25.31                25.59
3  Middle East & North Africa             -4.25           21.14                21.56
1         East Asia & Pacific             -3.27           17.28                17.59
5          Sub-Saharan Africa             -1.99           15.37                15.49
0                     America             -3.32           14.62                15.00
2       Europe & Central Asia             -1.53            8.27                 8.41


In [11]:
# plotting
fig = px.bar(
    pivot,
    x = "transition_distance",
    y = "Region",
    orientation = "h",
    text = "transition_distance",
    title = "Demographic Transition Distance by Continent (1964-2013)",
    labels = {
        "transition_distance": "Transition Distance",
        "Region": "Region",
    },
    hover_data = {
        "fertility_change": ":.2f",
        "lifeExp_change": ":.2f",
        "transition_distance": ":,2f",
    }
)

fig.update_traces(
    texttemplate = "%{text:.2f}",
    textposition = "outside"
)
fig.update_layout(height = 500, width = 600)
fig.show()

## Sub-Question 2: When did countries experience their demographic transition?

At what point did each country cross the threshold from high fertility and low life expectancy to low fertility and high life expectancy, and how did the timing differ across continents?

Defining a threshold which shows the year the country complete the transition is necessary. The demographic transition year is defined as the first year in which a country reaches replacement-level fertility (total fertility rate ≤ 2.1 children per woman) and a high level of survival (life expectancy at birth ≥ 70 years). Considering random fluctuations, both conditions should be remaining satisfied for at least five consecutive years.

In [37]:
def find_transition_year(group):
    condition = (
        (group["Fertility"] <= 2.1) &
        (group["lifeExp"] >= 70)
    )

    year = group.loc[condition, "Year"]

    if len(year) == 0:
        return np.nan

    return year.min()

transition = (
    df.groupby("Country")
      .apply(find_transition_year)
      .reset_index(name="Transition Year")
)

def decade_bin(year):
    if pd.isna(year):
        return "Not reached"
    elif year < 1970:
        return "1960s"
    elif year < 1980:
        return "1970s"
    elif year < 1990:
        return "1980s"
    elif year < 2000:
        return "1990s"
    elif year < 2010:
        return "2000s"
    else:
        return "2010s"

transition["Decades"] = (transition["Transition Year"].apply(decade_bin))

In [38]:
import pycountry

def get_iso3(country):
    try:
        return pycountry.countries.lookup(country).alpha_3
    except:
        return None

transition["ISO3"] = transition["Country"].apply(get_iso3)

In [43]:
fig = px.choropleth(
    transition,
    locations = "ISO3",
    locationmode = "ISO-3",
    color = "Decades",
    hover_name = "Country",
    hover_data = {"Transition Year":":.0f"},
    category_orders = {
        "Decades": [
            "1960s",
            "1970s",
            "1980s",
            "1990s",
            "2000s",
            "2010s",
            "Not reached"
        ]
    },
    color_discrete_map={
    "1960s": "#08306b",
    "1970s": "#2171b5",
    "1980s": "#6baed6",
    "1990s": "#fd8d3c",
    "2000s": "#e6550d",
    "2010s": "#a63603",
    "Not reached": "lightgray"
    },
    title = "World Demographic Transition Year Distribution",
    height = 700,
    width = 1000,
)

fig.show()